In [ ]:
import json, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

RULES = json.load(open('rules.json'))
BS = RULES['benign_stats']
SEED, N = 42, 3300
rng = np.random.default_rng(SEED)

ATTACKER, TARGET = 'anon_ip_21', RULES['pinned']['ip_b']
TIMEFRAMES = [5, 6, 7, 8, 9, 10, 11]
TF_WEIGHT  = {5:1, 6:1, 7:1, 8:1, 9:1, 10:1, 11:0.5}   # steady 5-10, winds down in 11
N_TARGET_PORTS = 5                                    # "several ports"
PSH_SHARE = (0.0, 1.0)                                # "some also with PSH"

In [ ]:
Q_PROBS = [0, .05, .25, .5, .75, .95, 1]

def benign_quantiles(col):
    s = BS[col]; return [s['min'], s['p5'], s['p25'], s['p50'], s['p75'], s['p95'], s['max']]

def sample_benign(col, n):
    """Sample from the benign distribution of a column (via its quantiles)."""
    return np.interp(rng.uniform(0, 1, n), Q_PROBS, benign_quantiles(col))

def log_uniform(lo, hi, n):
    return np.exp(rng.uniform(np.log(lo), np.log(hi), n))

In [ ]:
w = np.array([TF_WEIGHT[t] for t in TIMEFRAMES]); counts = np.floor(N * w / w.sum()).astype(int)
counts[0] += N - counts.sum()
df = pd.DataFrame({'timeframe_id': np.repeat(TIMEFRAMES, counts)})
n = len(df)

# ids / ports
df['proto'], df['ip_a'] = 6, ATTACKER
lo_p, hi_p = RULES['id_formats']['port_b_range']
target_ports = rng.choice(np.arange(max(1024, lo_p), hi_p + 1), N_TARGET_PORTS, replace=False)
df['port_b'] = rng.choice(target_ports, n)
df['port_a'] = 0
for t, idx in df.groupby('timeframe_id').groups.items():
    df.loc[idx, 'port_a'] = rng.choice(np.arange(1024, 65536), len(idx), replace=False)

# packets
df['pkts_fwd'] = np.round(log_uniform(BS['pkts_fwd']['min'], BS['pkts_fwd']['max'], n)).astype(int).clip(1)
p = df['pkts_fwd'].values

# packet sizes: large payloads -> upper quarter of benign size range
size_lo = BS['pkt_size_min']['min'] + 0.75 * (BS['pkt_size_max']['max'] - BS['pkt_size_min']['min'])
size_hi = BS['pkt_size_max']['max']
a, b = rng.uniform(size_lo, size_hi, n), rng.uniform(size_lo, size_hi, n)
smin, smax = np.round(np.minimum(a, b)), np.round(np.maximum(a, b))
smax = np.where(p == 1, smin, smax)
mean_size = smin + rng.uniform(0, 1, n) * (smax - smin)
df['pkt_size_min'], df['pkt_size_max'] = smin.astype(int), smax.astype(int)
df['bytes_fwd'] = np.clip(np.round(p * mean_size), p * smin, p * smax).astype(int)
df['pkt_size_std'] = np.where(p == 1, 0.0, rng.uniform(0, 0.5, n) * (smax - smin))
df['small_pkt_fraction_total'] = 0.0
df['tiny_pkt_fraction_total']  = 0.0
df['truncated_udp_fraction']   = 0.0

# flags
df['ack_count_fwd'] = p
df['psh_count_fwd'] = rng.binomial(p, rng.uniform(*PSH_SHARE, n))
for f in ['syn', 'fin', 'rst', 'urg']:
    df[f'{f}_count_fwd'] = 0

# timing: high steady stream -> benign p5-p25 IAT band, low spread
mean_iat = log_uniform(BS['mean_iat_int_ns']['p5'], BS['mean_iat_int_ns']['p25'], n)
iat_std = np.where(p == 2, 0.0, rng.uniform(0, 0.5, n) * mean_iat)
df['mean_iat_int_ns'] = mean_iat
df['iat_std_int_ns']  = iat_std
df['iat_min_int_ns']  = np.maximum(mean_iat - 2 * iat_std, 0.1 * mean_iat)
df['iat_max_int_ns']  = mean_iat + 2 * iat_std
df['duration'] = np.where(p == 1, 0.0, (p - 1) * mean_iat / 1e9)

# INT telemetry: path property -> benign distributions
int_missing = (p <= 2) & (rng.uniform(0, 1, n) < BS['mean_hop_latency']['empty_rate'])
df['mean_hop_count'] = 1.0
df['mean_hop_latency'] = sample_benign('mean_hop_latency', n)
l1, l2 = sample_benign('latency_min', n), sample_benign('latency_max', n)
df['latency_min'] = np.minimum.reduce([l1, l2, df['mean_hop_latency']])
df['latency_max'] = np.maximum.reduce([l1, l2, df['mean_hop_latency']])
df['latency_std'] = np.where(p == 1, 0.0, sample_benign('latency_std', n))
q_mean = sample_benign('queue_occupancy_mean', n)
df['queue_occupancy_mean'] = q_mean
df['queue_occupancy_max']  = np.where(p == 1, q_mean, np.maximum(q_mean, sample_benign('queue_occupancy_max', n)))
df['queue_occupancy_std']  = np.where(p == 1, 0.0, sample_benign('queue_occupancy_std', n))
df.loc[p == 1, ['latency_min', 'latency_max']] = np.c_[df.loc[p == 1, 'mean_hop_latency'], df.loc[p == 1, 'mean_hop_latency']]

INT_COLS = RULES['empty_rules'][1]['columns']; IAT_COLS = RULES['empty_rules'][2]['columns']
df.loc[int_missing, INT_COLS] = np.nan
df.loc[(p == 1) | int_missing, IAT_COLS] = np.nan

In [ ]:
for c, v in RULES['pinned'].items():
    df[c] = v
for c in RULES['always_empty']:
    df[c] = np.nan
df['pkts_total']  = df['pkts_fwd'] + df['pkts_bwd']
df['bytes_total'] = df['bytes_fwd'] + df['bytes_bwd']

# source_*: this attacker's flows in the same timeframe
g = df.groupby('timeframe_id')
df['source_flow_count']          = g['pkts_total'].transform('size')
df['source_pkts_total']          = g['pkts_total'].transform('sum')
df['source_bytes_total']         = g['bytes_total'].transform('sum')
df['source_unique_target_ips']   = 1
df['source_unique_target_ports'] = g['port_b'].transform('nunique')
df['source_unique_peer_ports']   = g['port_a'].transform('nunique')

# target_service_*: traffic to this host+port
gs = df.groupby(['timeframe_id', 'port_b'])
df['target_service_flow_count']        = gs['pkts_total'].transform('size')
df['target_service_unique_peer_ips']   = 1
df['target_service_unique_peer_ports'] = gs['port_a'].transform('nunique')
df['target_service_pkts_total']        = gs['pkts_total'].transform('sum')
df['target_service_bytes_total']       = gs['bytes_total'].transform('sum')

# target_ip_*: attack + normal background (one benign draw per timeframe)
BG_COLS = ['target_ip_flow_count', 'target_ip_unique_peer_ips', 'target_ip_unique_peer_ports',
           'target_ip_unique_services', 'target_ip_pkts_total', 'target_ip_bytes_total']
bg = {t: {c: np.round(sample_benign(c, 1)[0]) for c in BG_COLS} for t in TIMEFRAMES}
B = lambda c: df['timeframe_id'].map(lambda t: bg[t][c])
df['target_ip_flow_count']        = B('target_ip_flow_count') + df['source_flow_count']
df['target_ip_unique_peer_ips']   = B('target_ip_unique_peer_ips') + 1
df['target_ip_unique_peer_ports'] = B('target_ip_unique_peer_ports') + df['source_unique_peer_ports']
df['target_ip_unique_services']   = B('target_ip_unique_services') + df['source_unique_target_ports']
df['target_ip_pkts_total']        = B('target_ip_pkts_total') + df['source_pkts_total']
df['target_ip_bytes_total']       = B('target_ip_bytes_total') + df['source_bytes_total']

# the 42 formulas from rules.json
for c, expr in RULES['derived'].items():
    df[c] = df.eval(expr, engine='python')
rate_cols = RULES['empty_rules'][0]['columns']
df.loc[df['duration'] == 0, rate_cols] = np.nan

df['flow_key'] = df[['timeframe_id', 'proto', 'ip_a', 'ip_b', 'port_a', 'port_b']].astype(str).agg('|'.join, axis=1)
df['is_attack'] = True

for c, dt in RULES['dtypes'].items():
    if dt == 'int' and df[c].notna().all():
        df[c] = df[c].round().astype('int64')
heur = df[RULES['column_order']].copy()
print(heur.shape)

In [ ]:
h = heur
checks = {
 'pkts_fwd==1 <=> duration==0': ((h.pkts_fwd == 1) == (h.duration == 0)).all(),
 'rates NA <=> duration==0': (h[rate_cols].isna().all(axis=1) == (h.duration == 0)).all(),
 'INT all-or-none': (h[INT_COLS].isna().all(axis=1) == h[INT_COLS].isna().any(axis=1)).all(),
 'INT NA only if pkts<=2': (h[INT_COLS[0]].notna() | (h.pkts_fwd <= 2)).all(),
 'IAT NA if pkts==1': h.loc[h.pkts_fwd == 1, IAT_COLS].isna().all().all(),
 'IAT filled if pkts>=3': h.loc[h.pkts_fwd >= 3, IAT_COLS].notna().all().all(),
 'pkt min<=mean<=max': ((h.pkt_size_min <= h.mean_pkt_size + 1e-9) & (h.mean_pkt_size <= h.pkt_size_max + 1e-9)).all(),
 'iat min<=mean<=max': (((h.iat_min_int_ns <= h.mean_iat_int_ns) & (h.mean_iat_int_ns <= h.iat_max_int_ns)) | h.mean_iat_int_ns.isna()).all(),
 'lat min<=mean<=max': (((h.latency_min <= h.mean_hop_latency) & (h.mean_hop_latency <= h.latency_max)) | h.mean_hop_latency.isna()).all(),
 'flags <= pkts': h[['syn_count_fwd','ack_count_fwd','fin_count_fwd','rst_count_fwd','psh_count_fwd','urg_count_fwd']].le(h.pkts_fwd, axis=0).all().all(),
 'pkts<=service<=ip': ((h.pkts_total <= h.target_service_pkts_total) & (h.target_service_pkts_total <= h.target_ip_pkts_total)).all(),
 'pkts<=source<=ip': ((h.pkts_total <= h.source_pkts_total) & (h.source_pkts_total <= h.target_ip_pkts_total)).all(),
 'service_flows<=ip_flows': (h.target_service_flow_count <= h.target_ip_flow_count).all(),
 'no negatives': (h.select_dtypes('number').fillna(0) >= 0).all().all(),
 'unique flow_key': h.flow_key.is_unique,
}
for k, v in checks.items(): print(f"{'OK  ' if v else 'FAIL'}  {k}")
print('duplicates:', h.duplicated().sum())

out = f'heuristic_attack_N{N}_seed{SEED}.csv'
heur.to_csv(out, index=False)
from google.colab import files
files.download(out)